# Lab 3 - Instruction tuning : du texte brut à l'assistant
### Introduction aux LLMs - ESGI Paris | Séance 3
---
**Thème du lab :** transformer un petit modèle de langage brut en assistant qui suit des instructions
**Durée estimée :** 1h - 1h30
**Objectifs :**
- Explorer un dataset d'instructions (Alpaca)
- Fine-tuner (SFT) un petit modèle open-source avec `transformers` / `trl`
- Comparer qualitativement les réponses du modèle avant et après le SFT, avec une grille façon LLM-as-judge

**Prérequis :** Python 3.8+. Sur Google Colab : Exécution > Modifier le type d'exécution > GPU (T4) si disponible — le CPU fonctionne aussi, juste plus lentement.

**Modèle utilisé :** [`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), un petit modèle de base (135 millions de paramètres, **non** instruction-tuned) — assez petit pour être fine-tuné en quelques minutes sans crédits GPU.
**Dataset utilisé :** [`tatsu-lab/alpaca`](https://huggingface.co/datasets/tatsu-lab/alpaca), 52 000 instructions générées via Self-Instruct (voir séance 3, section 4).

## Installation des dépendances

In [1]:
# À exécuter une seule fois
import subprocess, sys

packages = ["transformers", "datasets", "trl", "accelerate", "torch", "matplotlib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("Installation terminée.")

Installation terminée.


---
## Partie 1 - Explorer un dataset d'instructions

Alpaca (Stanford, 2023) contient des triplets **(instruction, entrée optionnelle, réponse)**, générés automatiquement à partir de GPT-3 via la méthode **Self-Instruct** (voir séance 3, section 4).

In [2]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Nombre total d'exemples : {len(dataset)}")
print()

for i in range(3):
    print(f"--- Exemple {i} ---")
    print("Instruction :", dataset[i]["instruction"])
    if dataset[i]["input"]:
        print("Entrée      :", dataset[i]["input"])
    print("Réponse     :", dataset[i]["output"])
    print()

/opt/anaconda3/envs/reddit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 52002/52002 [00:00<00:00, 1181002.13 examples/s]

Nombre total d'exemples : 52002

--- Exemple 0 ---
Instruction : Give three tips for staying healthy.
Réponse     : 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.

--- Exemple 1 ---
Instruction : What are the three primary colors?
Réponse     : The three primary colors are red, blue, and yellow.

--- Exemple 2 ---
Instruction : Describe the structure of an atom.
Réponse     : An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative charge, resulting in an overall neutral atom. The number of each particle determines the atomic number and the type of atom.



### 1.1 Longueur des réponses

**Exercice :** Affichez un histogramme (`matplotlib`) du nombre de mots (`len(reponse.split())`) des réponses (`output`) sur les 2000 premiers exemples du dataset. Que remarquez-vous sur la distribution ?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

### 1.2 Repérer des exemples de qualité discutable

Les datasets générés automatiquement (Self-Instruct) ne sont pas parfaits : instructions ambiguës, réponses trop courtes, parfois incorrectes.

**Exercice :** Parcourez les 20 exemples ci-dessous et identifiez-en **au moins 2** qui vous semblent ambigus, incomplets ou faux. Justifiez votre choix dans la cellule markdown suivante.

In [ ]:
echantillon = dataset.shuffle(seed=42).select(range(20))

for i, ex in enumerate(echantillon):
    print(f"[{i}] Instruction : {ex['instruction']}")
    if ex["input"]:
        print(f"     Entrée : {ex['input']}")
    print(f"     Réponse : {ex['output']}")
    print()

**Votre réponse :** (indiquez les numéros d'exemples choisis et pourquoi)

---
## Partie 2 - SFT léger sur un petit modèle

On charge **deux copies** du même modèle de base : `model_before` restera intact (référence), `model_after` sera fine-tuné. Le modèle étant petit (135M paramètres), garder les deux en mémoire ne pose pas de problème.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_before = AutoModelForCausalLM.from_pretrained(model_name)
model_after = AutoModelForCausalLM.from_pretrained(model_name)

def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Modèles chargés.")

### 2.1 Le modèle brut, avant tout fine-tuning

On reprend l'exemple exact de la séance 3 (section 4) : un modèle pré-entraîné brut **continue du texte**, il ne répond pas à une instruction.

In [9]:
prompt_naked = "What is the capital of France?\n"

print("=== AVANT SFT (modèle brut) ===")
print(generate(model_before, prompt_naked))

=== AVANT SFT (modèle brut) ===
What is the capital of France?

The capital of France is Paris.

What is the capital of France?

The capital of France is Paris.

What is the capital of France?

The capital of France is Paris.

What is the capital of France?

The capital of France is


Le modèle continue probablement le texte (par exemple en enchaînant d'autres questions), plutôt que de répondre — exactement comme décrit en cours.

### 2.2 Préparer un petit sous-ensemble d'instructions

Pour que l'entraînement prenne quelques minutes seulement, on prend un sous-ensemble réduit (300 exemples), qu'on formate en un unique champ de texte `"### Instruction: ... ### Réponse: ..."`.

In [8]:
def format_example(example):
    if example["input"]:
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Entrée:\n{example['input']}\n\n### Réponse:\n"
        )
    else:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Réponse:\n"
    return {"text": prompt + example["output"] + tokenizer.eos_token}

train_subset = dataset.shuffle(seed=42).select(range(300))
train_subset = train_subset.map(format_example)

print(train_subset[0]["text"])

Map: 100%|██████████| 300/300 [00:00<00:00, 18337.63 examples/s]

### Instruction:
What would be the best type of exercise for a person who has arthritis?

### Réponse:
For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.<|endoftext|>


### 2.3 Fine-tuning avec `SFTTrainer` (librairie TRL)

`SFTTrainer` gère la tokenisation et l'entraînement supervisé à partir d'un simple champ `"text"`. On limite volontairement le nombre de pas (`max_steps`) pour que ça tienne dans la séance.

⚠️ La librairie `trl` évolue vite : si un paramètre ci-dessous n'est pas reconnu (ex. `max_length` vs `max_seq_length`), regardez `SFTConfig?` dans une cellule pour voir les noms à jour.

In [10]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./smollm2-alpaca-sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_steps=60,          # limite le temps d'entraînement pour le lab
    learning_rate=2e-5,
    max_length=256,
    dataset_text_field="text",
    logging_steps=10,
    report_to="none",
)

trainer = SFTTrainer(
    model=model_after,
    args=sft_config,
    train_dataset=train_subset,
)

trainer.train()

Truncating train dataset: 100%|██████████| 300/300 [00:00<00:00, 10593.82 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.
/opt/anaconda3/envs/reddit/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,2.078192
20,2.168937
30,2.135958
40,2.103084
50,2.184472
60,2.068134


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]


TrainOutput(global_step=60, training_loss=2.1231295267740884, metrics={'train_runtime': 61.9824, 'train_samples_per_second': 7.744, 'train_steps_per_second': 0.968, 'total_flos': 41819522462208.0, 'train_loss': 2.1231295267740884, 'epoch': 1.5866666666666667})

---
## Partie 3 - Comparer avant / après

### 3.1 Même prompt, avant et après SFT

In [11]:
prompt_template = f"### Instruction:\nExplain what photosynthesis is.\n\n### Réponse:\n"

print("=== AVANT SFT (prompt naturel) ===")
print(generate(model_before, prompt_naked))
print()
print("=== APRÈS SFT (prompt naturel, sans template) ===")
print(generate(model_after, prompt_naked))
print()
print("=== APRÈS SFT (prompt au format d'entraînement) ===")
print(generate(model_after, prompt_template))

=== AVANT SFT (prompt naturel) ===


/opt/anaconda3/envs/reddit/lib/python3.10/site-packages/transformers/generation/utils.py:2636: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on mps. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('mps') before running `.generate()`.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


What is the capital of France?

The capital of France is Paris.

What is the capital of France?

The capital of France is Paris.

What is the capital of France?

The capital of France is Paris.

What is the capital of France?

The capital of France is

=== APRÈS SFT (prompt naturel, sans template) ===


RuntimeError: Placeholder storage has not been allocated on MPS device!

**Observation attendue :** le modèle après SFT devrait produire une réponse plus courte et plus pertinente, surtout lorsqu'on utilise le même format (`### Instruction: ... ### Réponse:`) que celui vu à l'entraînement — ce qui montre que le modèle a appris un **format**, pas seulement des *(instruction, réponse)* précises.

### 3.2 À vous de jouer

**Exercice :** Choisissez 2 nouvelles instructions (différentes de celles vues à l'entraînement, en anglais pour rester dans la langue du dataset). Pour chacune, générez et comparez la réponse de `model_before` et de `model_after` (avec le prompt au format d'entraînement).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Bonus (optionnel) :** réessayez avec une instruction **en français**. Le résultat est-il aussi bon ? Pourquoi, en repensant au biais multilingue vu en section 1 (tokenization) et au fait qu'Alpaca est un dataset anglophone ?

### 3.3 Grille d'évaluation façon LLM-as-judge

Noter à la main chaque paire de réponses ne passe pas à l'échelle (cf. séance 3, section 6). En pratique, on utilise un LLM puissant comme **juge** (MT-Bench, Zheng et al., 2023). On construit ici le prompt de jugement — à copier-coller dans l'interface de chat d'un LLM (ChatGPT, Claude, Gemini...).

In [ ]:
def build_judge_prompt(instruction, reponse_a, reponse_b):
    return f"""Tu es un juge impartial. Voici une instruction et deux réponses générées par deux modèles différents.
Note chaque réponse de 1 à 10 sur les critères suivants : pertinence, respect de l'instruction, clarté.
Indique ensuite quelle réponse est la meilleure et pourquoi, en 2 phrases maximum.

Instruction : {instruction}

Réponse A : {reponse_a}

Réponse B : {reponse_b}
"""

reponse_avant = generate(model_before, prompt_naked)
reponse_apres = generate(model_after, prompt_template)

judge_prompt = build_judge_prompt(
    "Explain what photosynthesis is.",
    reponse_avant,
    reponse_apres,
)
print(judge_prompt)

**Exercice :** Copiez le texte affiché ci-dessus dans une interface de chat LLM (ou appelez une API si vous avez une clé). Notez le verdict, puis refaites l'opération avec vos 2 instructions de la section 3.2.

⚠️ **Biais connus du LLM-as-judge** (vus en séance 3, section 6) : préférence pour les réponses longues, pour un certain style, sensibilité à l'ordre de présentation (A avant B). Un bon réflexe : inverser l'ordre A/B et vérifier que le verdict ne change pas.

---
## Récapitulatif

Remplissez le tableau ci-dessous avec vos observations (oui / non / partiellement) :

| | Suit le format `Instruction → Réponse` | Réponse pertinente | Réponse concise |
|---|---|---|---|
| **Avant SFT** | ? | ? | ? |
| **Après SFT (prompt naturel)** | ? | ? | ? |
| **Après SFT (prompt template)** | ? | ? | ? |

**Pour aller plus loin (non couvert dans ce lab, voir séance 3 section 5) :** le SFT seul ne garantit pas un modèle *aligné* sur des préférences fines (ton, sécurité, utilité perçue) — c'est le rôle du RLHF ou de DPO, qui partent d'un modèle déjà instruction-tuned comme celui obtenu ici.